# Attention Mechanisms — Advanced
> Section 01 — Topic 03 — advanced

**Prerequisites:** 03-attention-mechanisms/foundations.ipynb

**What you'll learn:**
- Modern efficient attention variants used in production LLMs
- Multi-Query Attention (MQA) and Grouped-Query Attention (GQA)
- Flash Attention's IO-aware algorithm
- Sliding window attention for long sequences

**What you'll build:**
- Benchmarks comparing all attention variants on speed and memory

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
import time

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")

## 1. Multi-Query Attention (MQA) — Shared KV Heads

Standard multi-head attention (MHA) uses separate key and value projections for each head. This means the KV cache — which stores past key/value pairs for efficient autoregressive generation — grows linearly with the number of heads. For a model with 32 heads and a 4096-token context, this cache consumes enormous GPU memory.

**Multi-Query Attention** (MQA), introduced by Shazeer (2019), solves this by using a **single shared** key and value head across all query heads. Each query head still has its own projection, so the model maintains expressiveness in what it's looking for, but all heads share the same key-value pairs. This reduces KV cache size by a factor equal to the number of heads.

The tradeoff is a slight quality degradation — since all heads look at the same keys and values, there's less diversity in what information each head extracts. However, the memory savings are dramatic: for a 32-head model, MQA uses 32x less KV cache memory. This directly translates to being able to serve longer contexts and more concurrent requests.

In [ ]:
class MultiQueryAttention(nn.Module):
    """Multi-Query Attention: single KV head shared across all query heads."""
    
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.d_model = d_model
        
        # Each query head gets its own projection
        self.W_q = nn.Linear(d_model, d_model, bias=False)  # n_heads * d_k
        # Single shared key and value projection
        self.W_k = nn.Linear(d_model, self.d_k, bias=False)  # Only 1 head worth
        self.W_v = nn.Linear(d_model, self.d_k, bias=False)  # Only 1 head worth
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        
        # Project queries: (batch, seq, d_model) -> (batch, n_heads, seq, d_k)
        Q = self.W_q(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        
        # Project single key/value: (batch, seq, d_k)
        K = self.W_k(x)  # (batch, seq, d_k)
        V = self.W_v(x)  # (batch, seq, d_k)
        
        # Expand K, V to all heads: (batch, 1, seq, d_k) -> broadcast with Q
        K = K.unsqueeze(1)  # (batch, 1, seq, d_k)
        V = V.unsqueeze(1)  # (batch, 1, seq, d_k)
        
        # Attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, V)  # V broadcasts across heads
        
        # Concatenate heads
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_o(attn_output), attn_weights

# Compare parameter counts
d_model, n_heads = 512, 8

mha_params = 4 * d_model * d_model  # Q, K, V, O projections
mqa_params = d_model * d_model + 2 * d_model * (d_model // n_heads) + d_model * d_model  # Q, K(1 head), V(1 head), O

print(f"MHA parameters: {mha_params:,}")
print(f"MQA parameters: {mqa_params:,}")
print(f"MQA saves {(1 - mqa_params/mha_params)*100:.1f}% of attention parameters")
print(f"\nKV cache per token:")
print(f"  MHA: {n_heads * 2 * (d_model // n_heads) * 2} bytes (fp16)")
print(f"  MQA: {2 * (d_model // n_heads) * 2} bytes (fp16)")
print(f"  MQA cache is {n_heads}x smaller")

## 2. Grouped-Query Attention (GQA) — The Middle Ground

GQA, introduced in the Llama 2 paper (Ainslie et al., 2023), is a compromise between MHA and MQA. Instead of every head having its own KV projections (MHA) or all heads sharing one (MQA), GQA divides query heads into **groups**, where each group shares a single set of key-value heads.

For example, with 8 query heads and 2 KV groups, heads 0-3 share one KV pair and heads 4-7 share another. This gives a 4x reduction in KV cache (vs the 8x from MQA) while retaining more expressiveness than MQA.

GQA is the dominant approach in modern models: Llama 2 70B, Llama 3, Mistral, and most production LLMs use GQA. It's the sweet spot between quality and efficiency.

In [ ]:
class GroupedQueryAttention(nn.Module):
    """Grouped-Query Attention: groups of query heads share KV heads."""
    
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int):
        super().__init__()
        assert n_heads % n_kv_heads == 0, "n_heads must be divisible by n_kv_heads"
        
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_groups = n_heads // n_kv_heads  # queries per KV head
        self.d_k = d_model // n_heads
        self.d_model = d_model
        
        self.W_q = nn.Linear(d_model, n_heads * self.d_k, bias=False)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        
        Q = self.W_q(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.n_kv_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.n_kv_heads, self.d_k).transpose(1, 2)
        
        # Expand KV heads to match query heads by repeating
        # (batch, n_kv_heads, seq, d_k) -> (batch, n_heads, seq, d_k)
        K = K.repeat_interleave(self.n_groups, dim=1)
        V = V.repeat_interleave(self.n_groups, dim=1)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, V)
        
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_o(attn_output), attn_weights

# Compare all three variants
d_model, n_heads = 512, 8
configs = {
    "MHA (8 KV heads)": GroupedQueryAttention(d_model, n_heads, n_kv_heads=8),
    "GQA (2 KV heads)": GroupedQueryAttention(d_model, n_heads, n_kv_heads=2),
    "MQA (1 KV head)":  GroupedQueryAttention(d_model, n_heads, n_kv_heads=1),
}

x = torch.randn(1, 32, d_model)
for name, module in configs.items():
    n_params = sum(p.numel() for p in module.parameters())
    output, _ = module(x)
    print(f"{name}: {n_params:,} params, output {output.shape}")

## 3. Flash Attention — The Memory-Efficient Algorithm

Flash Attention (Dao et al., 2022) is one of the most impactful optimizations in modern deep learning. It doesn't change what attention computes — the mathematical result is identical — but it changes **how** the computation is performed to dramatically reduce memory usage and improve speed.

The key insight is about **IO complexity**. Standard attention computes the full $N \times N$ attention matrix, writes it to GPU HBM (high-bandwidth memory), then reads it back to multiply with V. For long sequences, this intermediate matrix is huge and the memory reads/writes (IO) dominate the runtime, not the actual FLOPs.

Flash Attention uses **tiling**: it processes attention in blocks, computing partial softmax results on-chip in fast SRAM and never materializing the full attention matrix in HBM. This requires a clever online softmax algorithm that can update running statistics as new blocks are processed.

The result: Flash Attention uses $O(N)$ memory instead of $O(N^2)$, and is 2-4x faster despite performing slightly more FLOPs, because it's IO-efficient. This is available in PyTorch as `torch.nn.functional.scaled_dot_product_attention` with the `flash` backend.

In [ ]:
# Standard attention — materializes full N×N matrix
def standard_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # O(N^2) memory
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)  # O(N^2) memory
    return torch.matmul(weights, V)

# Using PyTorch's optimized SDPA (uses Flash Attention when available)
def pytorch_sdpa(Q, K, V, is_causal=False):
    return F.scaled_dot_product_attention(Q, K, V, is_causal=is_causal)

# Verify they produce the same result
batch, heads, seq_len, d_k = 2, 8, 64, 64
Q = torch.randn(batch, heads, seq_len, d_k)
K = torch.randn(batch, heads, seq_len, d_k)
V = torch.randn(batch, heads, seq_len, d_k)

out_standard = standard_attention(Q, K, V)
out_sdpa = pytorch_sdpa(Q, K, V)

print(f"Max difference: {(out_standard - out_sdpa).abs().max().item():.2e}")
print(f"Results are {'equivalent' if (out_standard - out_sdpa).abs().max() < 1e-5 else 'different'}")

# Check which backends are available
print(f"\nSDPA backends available:")
print(f"  Flash Attention: {torch.backends.cuda.flash_sdp_enabled() if torch.cuda.is_available() else 'N/A (no CUDA)'}")
print(f"  Memory-efficient: {torch.backends.cuda.mem_efficient_sdp_enabled() if torch.cuda.is_available() else 'N/A (no CUDA)'}")
print(f"  Math fallback: always available")

## 4. Ring Attention — Distributed Attention Across Devices

Even with Flash Attention, a single GPU has limited memory for the KV cache, which constrains the maximum context length. **Ring Attention** (Liu et al., 2023) extends context length across multiple GPUs by distributing the attention computation in a ring topology.

The idea: split the input sequence into chunks, one per device. Each device computes attention for its chunk of queries against all key-value chunks. The KV blocks are passed around the ring — each device sends its KV block to the next device and receives from the previous one, computing partial attention results as blocks arrive. After one full rotation, every device has computed attention against all KV pairs.

This overlaps communication with computation: while a device is computing attention against the current KV block, it's simultaneously sending/receiving the next block. This makes Ring Attention nearly as fast as local attention while enabling context lengths proportional to the number of devices. Combined with Flash Attention for on-device computation, this enables million-token contexts.

In [ ]:
# Conceptual simulation of Ring Attention (single-device pseudocode)
# In practice, this runs across multiple GPUs with NCCL communication

def simulate_ring_attention(Q_chunks, K_chunks, V_chunks, n_devices):
    """
    Simulate ring attention on a single device.
    Each 'device' holds a chunk of Q and rotates K, V around the ring.
    """
    d_k = Q_chunks[0].shape[-1]
    results = []
    
    for device_id in range(n_devices):
        Q_local = Q_chunks[device_id]  # This device's queries
        
        # Accumulate attention across all KV blocks (ring rotation)
        weighted_sum = torch.zeros_like(Q_local)
        log_sum_exp = torch.full((Q_local.shape[0], Q_local.shape[1], 1), float('-inf'))
        
        for step in range(n_devices):
            # Receive KV block from ring position
            kv_source = (device_id + step) % n_devices
            K_block = K_chunks[kv_source]
            V_block = V_chunks[kv_source]
            
            # Compute local attention for this block
            scores = torch.matmul(Q_local, K_block.transpose(-2, -1)) / math.sqrt(d_k)
            block_max = scores.max(dim=-1, keepdim=True).values
            
            # Online softmax update (numerically stable accumulation)
            new_max = torch.maximum(log_sum_exp, block_max)
            exp_scores = torch.exp(scores - new_max)
            exp_old = torch.exp(log_sum_exp - new_max)
            
            weighted_sum = weighted_sum * exp_old + torch.matmul(exp_scores, V_block)
            log_sum_exp = new_max + torch.log(exp_old + exp_scores.sum(dim=-1, keepdim=True))
        
        results.append(weighted_sum / torch.exp(log_sum_exp))
    
    return torch.cat(results, dim=1)

# Demo: split sequence across 4 simulated devices
seq_len, d_k, n_devices = 16, 32, 4
chunk_size = seq_len // n_devices

Q = torch.randn(1, seq_len, d_k)
K = torch.randn(1, seq_len, d_k)
V = torch.randn(1, seq_len, d_k)

Q_chunks = [Q[:, i*chunk_size:(i+1)*chunk_size] for i in range(n_devices)]
K_chunks = [K[:, i*chunk_size:(i+1)*chunk_size] for i in range(n_devices)]
V_chunks = [V[:, i*chunk_size:(i+1)*chunk_size] for i in range(n_devices)]

ring_output = simulate_ring_attention(Q_chunks, K_chunks, V_chunks, n_devices)
standard_output = standard_attention(Q, K, V)

print(f"Ring attention output shape: {ring_output.shape}")
print(f"Max difference from standard: {(ring_output - standard_output).abs().max().item():.2e}")
print(f"\nIn production: each chunk lives on a separate GPU.")
print(f"With {n_devices} GPUs, context length scales {n_devices}x.")

## 5. Sliding Window Attention — Mistral's Approach

Sliding Window Attention (SWA), used prominently in Mistral 7B, is a simple but effective way to handle long sequences. Instead of attending to all previous tokens, each token only attends to a fixed **window** of the most recent $W$ tokens.

The key insight is that most useful information is local — a token rarely needs to attend to something thousands of tokens ago. And even though each layer only sees $W$ tokens, information can propagate through the network: after $L$ layers, a token effectively has access to $L \times W$ tokens of context through indirect connections.

For Mistral with $W = 4096$ and $L = 32$ layers, the theoretical receptive field is $32 \times 4096 = 131,072$ tokens. The memory savings are substantial: KV cache is bounded at $W$ entries regardless of sequence length, making inference memory predictable and constant.

In [ ]:
def sliding_window_attention(Q, K, V, window_size: int):
    """Attention where each query only attends to the nearest window_size keys."""
    batch, seq_len, d_k = Q.shape
    
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    # Create sliding window mask
    mask = torch.zeros(seq_len, seq_len, dtype=torch.bool)
    for i in range(seq_len):
        start = max(0, i - window_size + 1)
        mask[i, start:i+1] = True  # Causal + window
    
    scores = scores.masked_fill(~mask.unsqueeze(0), float('-inf'))
    weights = F.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)
    
    return output, weights, mask

# Demo: compare full causal vs sliding window
seq_len, d_k, window = 16, 32, 4
Q = torch.randn(1, seq_len, d_k)
K = torch.randn(1, seq_len, d_k)
V = torch.randn(1, seq_len, d_k)

out_sw, weights_sw, mask_sw = sliding_window_attention(Q, K, V, window)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Full causal mask
causal_mask = torch.tril(torch.ones(seq_len, seq_len))
axes[0].imshow(causal_mask.numpy(), cmap='Greens')
axes[0].set_title(f'Full Causal Attention\n(attends to all prior tokens)')

# Sliding window mask
axes[1].imshow(mask_sw.float().numpy(), cmap='Greens')
axes[1].set_title(f'Sliding Window (w={window})\n(attends to {window} nearest tokens)')

for ax in axes:
    ax.set_xlabel('Key Position')
    ax.set_ylabel('Query Position')

plt.tight_layout()
plt.show()

full_ops = seq_len * (seq_len + 1) // 2
sw_ops = seq_len * window
print(f"Full causal: ~{full_ops} attention ops")
print(f"Sliding window: ~{sw_ops} attention ops")
print(f"Sliding window uses {sw_ops/full_ops*100:.0f}% of the compute")

## 6. Benchmarking — Compare All Attention Variants

Let's put all the variants head-to-head with actual timing measurements. We'll compare standard MHA, MQA, GQA, and sliding window attention on speed and memory across different sequence lengths. This is the kind of benchmarking you'd do when choosing an architecture for a new model or optimizing inference.

In [ ]:
def benchmark_attention(attn_fn, Q, K, V, n_warmup=3, n_runs=10, **kwargs):
    """Benchmark an attention function."""
    # Warmup
    for _ in range(n_warmup):
        _ = attn_fn(Q, K, V, **kwargs)
    
    # Timed runs
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        _ = attn_fn(Q, K, V, **kwargs)
        times.append(time.perf_counter() - start)
    
    return np.median(times) * 1000  # ms

# Benchmark across sequence lengths
seq_lengths = [64, 128, 256, 512, 1024, 2048]
d_k = 64

results = {'Standard': [], 'Sliding Window (w=128)': [], 'PyTorch SDPA': []}

for seq_len in seq_lengths:
    Q = torch.randn(1, seq_len, d_k)
    K = torch.randn(1, seq_len, d_k)
    V = torch.randn(1, seq_len, d_k)
    
    results['Standard'].append(benchmark_attention(standard_attention, Q, K, V))
    results['Sliding Window (w=128)'].append(
        benchmark_attention(lambda q, k, v: sliding_window_attention(q, k, v, 128)[0], Q, K, V)
    )
    
    # PyTorch SDPA (reshape for multi-head format)
    Q_mh = Q.unsqueeze(1)  # Add head dim
    K_mh = K.unsqueeze(1)
    V_mh = V.unsqueeze(1)
    results['PyTorch SDPA'].append(
        benchmark_attention(lambda q, k, v: F.scaled_dot_product_attention(q, k, v), Q_mh, K_mh, V_mh)
    )

# Plot results
fig, ax = plt.subplots(figsize=(10, 6))
for name, times in results.items():
    ax.plot(seq_lengths, times, 'o-', label=name, linewidth=2)

ax.set_xlabel('Sequence Length')
ax.set_ylabel('Time (ms)')
ax.set_title('Attention Variant Benchmark (CPU)')
ax.legend()
ax.set_yscale('log')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Memory comparison: KV cache size per token for different variants
d_model = 4096  # Typical for 7B models
n_layers = 32
n_heads = 32
d_k = d_model // n_heads  # 128

configs = {
    "MHA (32 KV heads)": 32,
    "GQA (8 KV heads)": 8,
    "GQA (4 KV heads)": 4,
    "MQA (1 KV head)": 1,
}

context_lengths = [2048, 4096, 8192, 32768, 131072]

print(f"KV Cache Memory (GB) — {n_layers} layers, d_model={d_model}, fp16")
print(f"{'Config':<25} " + " ".join(f"{cl:>8}" for cl in context_lengths))
print("-" * 75)

memory_data = {}
for name, n_kv in configs.items():
    memories = []
    for ctx_len in context_lengths:
        # 2 (K and V) * n_kv_heads * d_k * n_layers * ctx_len * 2 (fp16 bytes)
        mem_bytes = 2 * n_kv * d_k * n_layers * ctx_len * 2
        mem_gb = mem_bytes / (1024**3)
        memories.append(mem_gb)
    memory_data[name] = memories
    print(f"{name:<25} " + " ".join(f"{m:>7.2f}G" for m in memories))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
for name, mems in memory_data.items():
    ax.plot(context_lengths, mems, 'o-', label=name, linewidth=2)

ax.set_xlabel('Context Length (tokens)')
ax.set_ylabel('KV Cache Memory (GB)')
ax.set_title('KV Cache Memory: MHA vs GQA vs MQA')
ax.legend()
ax.set_xscale('log')
ax.grid(True, alpha=0.3)
ax.axhline(y=24, color='red', linestyle='--', alpha=0.5, label='24GB GPU')
plt.tight_layout()
plt.show()

## Summary

**Key takeaways:**
- **MQA** uses a single shared KV head — maximum cache savings but some quality loss
- **GQA** groups query heads to share KV — the production sweet spot (Llama 2/3, Mistral)
- **Flash Attention** is an IO-aware algorithm that computes exact attention with O(N) memory — it's strictly better and should always be used
- **Ring Attention** distributes attention across GPUs for multi-million token contexts
- **Sliding Window Attention** bounds KV cache at window size — works because information propagates through layers

**Production models combine these:** Mistral uses GQA + Sliding Window + Flash Attention. Llama 3 uses GQA + Flash Attention.

**Next:** [03-attention-mechanisms/expert.ipynb](expert.ipynb) — Attention sinks, sparse attention, linear attention